# Creating Route Timing Pickle

This notebook builds a per-route stop sequence table with estimated travel times between consecutive stops, saved as route_stop_sequences.pkl for use in downstream imputation.

Stop Matching

- Raw GPS bus observations are matched to their nearest scheduled stop using a spatial lookup (cKDTree)
- Each bus ping is matched only to stops actually served by that route, within a - radius of 0.0002 degrees (~22m)
- This radius was chosen to balance GPS accuracy against the risk of false matches at nearby stops


Empirical Travel Times
- Consecutive stop arrivals from the same vehicle are paired to compute observed travel times
- These are averaged across all vehicles and trips to produce a mean travel time (t_avg_min) per stop pair
- Observations outside 0.75–30 minutes are discarded as implausible


Stop Sequence Construction
- For each route, the most complete trip in GTFS is used as the canonical stop sequence
- For each consecutive stop pair, travel time is assigned in order of preference:
    - Empirical — observed GPS average, if available and within bounds
    - Interpolated — if a run of stop pairs lacks empirical data but is flanked by empirical values on both sides, scheduled times are used to distribute the total gap proportionally between the two anchors
    - Scheduled — raw GTFS time, used only when no empirical anchors exist nearby
- This ensures absolute travel times are grounded in real observations wherever possible, with the schedule used only to inform relative proportions rather than absolute values


In [61]:
### IMPORTS
import matplotlib.pyplot as plt
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
import pickle

### Stop Matching

- Raw GPS bus observations are matched to their nearest scheduled stop using a spatial lookup (cKDTree)
- Each bus ping is matched only to stops actually served by that route, within a - radius of 0.0002 degrees (~22m)
- This radius was chosen to balance GPS accuracy against the risk of false matches at nearby stops

In [75]:
# Load bus data
#bus_df = pd.read_csv("missing_data_imputation/bus_data_after_missing.csv")
bus_df = pd.read_csv("data/all_data_final.csv")
bus_df["rt"] = bus_df["rt"].astype(str).str.strip()

# Load stops
stops_df = pd.read_csv("Datasets/stops.txt")[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

# Load trips and stop_times, filtering to only routes in bus_df
trips_df = pd.read_csv("Datasets/trips.txt")[["route_id", "trip_id"]]
trips_df["route_id"] = trips_df["route_id"].astype(str).str.strip()

relevant_routes = set(bus_df["rt"])
relevant_trips = set(trips_df[trips_df["route_id"].isin(relevant_routes)]["trip_id"])

# Only load the columns we need from stop_times
stop_times_df = pd.read_csv("Datasets/stop_times.txt")[["trip_id", "stop_id"]]
stop_times_df = stop_times_df[stop_times_df["trip_id"].isin(relevant_trips)]

# Build route_id -> set of valid stop_ids
trip_to_route = trips_df.set_index("trip_id")["route_id"].to_dict()
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_to_route)

route_to_stops = (
    stop_times_df.groupby("route_id")["stop_id"]
    .apply(set)
    .to_dict()
)

# Build cKDTree once from stops_df
stop_coords = stops_df[["stop_lat", "stop_lon"]].values
stop_tree = cKDTree(stop_coords)

def assign_stop_id(bus_lat, bus_lon, route_id, radius=0.0002):
    indices = stop_tree.query_ball_point([bus_lat, bus_lon], r=radius)

    if not indices:
        return None

    # Filter to stops actually served by this route
    valid_stops_for_route = route_to_stops.get(route_id, set())
    valid_indices = [
        i for i in indices
        if stops_df.iloc[i]["stop_id"] in valid_stops_for_route
    ]

    if not valid_indices:
        return None

    # Return nearest valid stop
    if len(valid_indices) > 1:
        candidate_coords = stop_coords[valid_indices]
        distances = np.linalg.norm(candidate_coords - np.array([bus_lat, bus_lon]), axis=1)
        nearest_idx = valid_indices[np.argmin(distances)]
    else:
        nearest_idx = valid_indices[0]

    return stops_df.iloc[nearest_idx]["stop_id"]

# Apply
bus_with_stops_df = bus_df.copy()
bus_with_stops_df["stop_id"] = bus_with_stops_df.apply(
    lambda row: assign_stop_id(row["lat"], row["lon"], row["rt"]), axis=1
)

matched = bus_with_stops_df["stop_id"].notna().sum()
total = len(bus_with_stops_df)
print(f"Matched {matched}/{total} rows ({matched/total:.1%}) to a stop")

bus_with_stops_df.head()


/var/folders/hh/2dsrbb3d021_w0zs6jdygn480000gn/T/ipykernel_39842/988840774.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bus_df = pd.read_csv("data/all_data_final.csv")


Matched 3062076/8234861 rows (37.2%) to a stop


,vid,tmstmp,lat,lon,hdg,pid,rt,des,pdist,dly,...,origtatripno,tablockid,zone,mode,psgld,stst,stsd,pulled_at,rt_chunk,stop_id
0,1095,20260220 18:41,41.878025,-87.641502,88,6351,1,34th/Michigan,2194,False,...,273982273.0,1 -757,NaN,1,NaN,66960,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
1,8473,20260220 18:41,41.847638,-87.623718,178,6351,1,34th/Michigan,18066,False,...,273982272.0,1 -756,NaN,1,NaN,65820,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
2,7931,20260220 18:41,41.866714,-87.624077,357,8085,1,Union Station,13113,False,...,273982246.0,1 -758,NaN,1,NaN,66360,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
3,1082,20260220 18:41,41.874788,-87.644205,207,8085,1,Union Station,24853,False,...,273982245.0,1 -755,NaN,1,NaN,65100,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
4,8692,20260220 18:41,41.883796,-87.627892,180,5528,2,Cottage Grove/60th,9162,False,...,273985972.0,2 -753,NaN,1,NaN,66000,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN


In [74]:
# have to test and evaluate radius values

for radius in [0.0001, 0.0002, 0.0003]: # this corresponds to  about 11m, 33m, 55m, 111m
    matched = bus_with_stops_df["rt"].apply(
        lambda rt: True  # placeholder
    )

    test = bus_df.copy()
    test["stop_id"] = test.apply(
        lambda row: assign_stop_id(row["lat"], row["lon"], row["rt"], radius=radius), axis=1
    )
    matched = test["stop_id"].notna().sum()
    total = len(test)
    print(f"radius={radius} ({radius*111000:.0f}m): {matched}/{total} ({matched/total:.1%}) matched")

radius=0.0001 (11m): 1300221/8234861 (15.8%) matched
radius=0.0002 (22m): 3062076/8234861 (37.2%) matched
radius=0.0003 (33m): 4220836/8234861 (51.3%) matched


Will use radius = 0.0002 (22m) going forward. Approximately 1/3 of buses will be considered at stops. 

### Empirical Travel Times
- Consecutive stop arrivals from the same vehicle are paired to compute observed travel times
- These are averaged across all vehicles and trips to produce a mean travel time (t_avg_min) per stop pair
- Observations outside 0.75–30 minutes are discarded as implausible

In [101]:
# Sort by route, vehicle, and timestamp
bus_with_stops_df = bus_with_stops_df.sort_values(["rt", "vid", "tmstmp"])

# Only keep rows that matched a stop
at_stops = bus_with_stops_df.dropna(subset=["stop_id"]).copy()

# For each vehicle trip, pair consecutive stop arrivals
at_stops["stop_i"] = at_stops.groupby(["rt", "vid"])["stop_id"].shift(0)
at_stops["stop_j"] = at_stops.groupby(["rt", "vid"])["stop_id"].shift(-1)
at_stops["time_i"] = pd.to_datetime(at_stops["tmstmp"])
at_stops["time_j"] = at_stops.groupby(["rt", "vid"])["time_i"].shift(-1)

at_stops["t_min"] = (at_stops["time_j"] - at_stops["time_i"]).dt.total_seconds() / 60

# Filter to valid consecutive stop pairs (same vehicle, reasonable travel time)
pairs = at_stops.dropna(subset=["stop_j", "t_min"])
pairs = pairs[(pairs["t_min"] > 0.5) & (pairs["t_min"] < 30)]

# Average travel time per route + stop pair
t_avg = (
    pairs.groupby(["rt", "stop_i", "stop_j"])["t_min"]
    .mean()
    .reset_index()
    .rename(columns={"rt": "route_id", "t_min": "t_avg_min"})
)

print(f"Computed t_avg for {len(t_avg)} route/stop-pair combinations")

Computed t_avg for 337586 route/stop-pair combinations


### Stop Sequence Construction
- For each route, the most complete trip in GTFS is used as the canonical stop sequence
- For each consecutive stop pair, travel time is assigned in order of preference:
    - Empirical — observed GPS average, if available and within bounds
    - Interpolated — if a run of stop pairs lacks empirical data but is flanked by empirical values on both sides, scheduled times are used to distribute the total gap proportionally between the two anchors
    - Scheduled — raw GTFS time, used only when no empirical anchors exist nearby
- This ensures absolute travel times are grounded in real observations wherever possible, with the schedule used only to inform relative proportions rather than absolute values


In [76]:
# Reload stop_times with stop_sequence this time
stop_times_df = pd.read_csv("Datasets/stop_times.txt")[["trip_id", "stop_id", "stop_sequence", "arrival_time"]]
stop_times_df = stop_times_df[stop_times_df["trip_id"].isin(relevant_trips)]

# Map route_id onto stop_times
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_to_route)

# Build the canonical stop sequence per route
# Use the most common trip as the representative sequence for each route
def get_canonical_sequence(group):
    # Find the trip_id with the most stops (most complete trip)
    best_trip = (
        group.groupby("trip_id")["stop_sequence"]
        .count()
        .idxmax()
    )
    return (
        group[group["trip_id"] == best_trip]
        .sort_values("stop_sequence")[["stop_sequence", "stop_id"]]
        .reset_index(drop=True)
    )

route_sequences = {}
for route_id, group in stop_times_df.groupby("route_id"):
    if route_id in relevant_routes:
        route_sequences[route_id] = get_canonical_sequence(group)

# Preview
sample_route = list(route_sequences.keys())[0]
print(f"Route {sample_route} has {len(route_sequences[sample_route])} stops in sequence")
print(route_sequences[sample_route].head())

Route 1 has 35 stops in sequence
   stop_sequence  stop_id
0              1    13155
1              2    18661
2              3    18498
3              4       67
4              5    14461


In [98]:
route_stop_sequences = {}

for route_id, group in stop_times_df.groupby("route_id"):
    if route_id not in relevant_routes:
        continue
    
    best_trip = group.groupby("trip_id")["stop_sequence"].count().idxmax()
    trip = group[group["trip_id"] == best_trip].sort_values("stop_sequence").reset_index(drop=True)
    trip["arrival_time"] = pd.to_timedelta(trip["arrival_time"])
    trip["t_scheduled_min"] = (
        trip["arrival_time"].shift(-1) - trip["arrival_time"]
    ).dt.total_seconds() / 60

    route_t_avg = t_avg[t_avg["route_id"] == route_id]

    records = []
    for i in range(len(trip) - 1):
        stop_i = trip.iloc[i]["stop_id"]
        stop_j = trip.iloc[i + 1]["stop_id"]
        t_sched = trip.iloc[i]["t_scheduled_min"]

        pair = route_t_avg[
            (route_t_avg["stop_i"] == stop_i) &
            (route_t_avg["stop_j"] == stop_j)
        ]
        t_empirical = pair["t_avg_min"].values[0] if not pair.empty else np.nan

        if not np.isnan(t_empirical) and 0.75 <= t_empirical <= 30:
            t_final = t_empirical
            source = "empirical"
        else:
            t_final = t_sched
            source = "scheduled"

        records.append({
            "stop_sequence": i,
            "stop_id": stop_i,
            "next_stop_id": stop_j,
            "t_avg_to_next_min": t_final,
            "source": source
        })

    records.append({
        "stop_sequence": len(trip) - 1,
        "stop_id": trip.iloc[-1]["stop_id"],
        "next_stop_id": None,
        "t_avg_to_next_min": np.nan,
        "source": None
    })

    seq_df = pd.DataFrame(records)

    # --- Interpolation pass ---
    # For runs of scheduled stops flanked by empirical anchors on both sides,
    # replace scheduled values with proportionally scaled empirical estimates
    empirical_mask = seq_df["source"] == "empirical"

    i = 0
    while i < len(seq_df):
        if seq_df.iloc[i]["source"] == "scheduled":
            # Find the end of this scheduled run
            j = i
            while j < len(seq_df) and seq_df.iloc[j]["source"] == "scheduled":
                j += 1

            # Look for empirical anchors immediately before and after
            prev_emp = seq_df[(seq_df.index < i) & empirical_mask]
            next_emp = seq_df[(seq_df.index >= j) & empirical_mask]

            if not prev_emp.empty and not next_emp.empty:
                total_sched = seq_df.iloc[i:j]["t_avg_to_next_min"].sum()

                if total_sched > 0:
                    # Estimate the total real time for this gap using neighboring empiricals
                    gap_estimate = (
                        prev_emp.iloc[-1]["t_avg_to_next_min"] +
                        next_emp.iloc[0]["t_avg_to_next_min"]
                    ) / 2

                    for k in range(i, j):
                        proportion = seq_df.iloc[k]["t_avg_to_next_min"] / total_sched
                        seq_df.at[k, "t_avg_to_next_min"] = proportion * gap_estimate
                        seq_df.at[k, "source"] = "interpolated"

            i = j  # skip past this run
        else:
            i += 1

    route_stop_sequences[route_id] = seq_df

# Diagnostics
for route_id, seq_df in route_stop_sequences.items():
    emp = (seq_df["source"] == "empirical").sum()
    interp = (seq_df["source"] == "interpolated").sum()
    sched = (seq_df["source"] == "scheduled").sum()
    total = emp + interp + sched
    print(f"Route {route_id}: {emp}/{total} empirical ({emp/total:.1%}), "
          f"{interp}/{total} interpolated ({interp/total:.1%}), "
          f"{sched}/{total} scheduled ({sched/total:.1%})")

Route 1: 21/34 empirical (61.8%), 13/34 interpolated (38.2%), 0/34 scheduled (0.0%)
Route 100: 15/49 empirical (30.6%), 34/49 interpolated (69.4%), 0/49 scheduled (0.0%)
Route 103: 17/62 empirical (27.4%), 45/62 interpolated (72.6%), 0/62 scheduled (0.0%)
Route 106: 12/25 empirical (48.0%), 12/25 interpolated (48.0%), 1/25 scheduled (4.0%)
Route 108: 19/44 empirical (43.2%), 17/44 interpolated (38.6%), 8/44 scheduled (18.2%)
Route 11: 21/32 empirical (65.6%), 11/32 interpolated (34.4%), 0/32 scheduled (0.0%)
Route 111: 23/44 empirical (52.3%), 21/44 interpolated (47.7%), 0/44 scheduled (0.0%)
Route 111A: 11/12 empirical (91.7%), 1/12 interpolated (8.3%), 0/12 scheduled (0.0%)
Route 112: 22/48 empirical (45.8%), 26/48 interpolated (54.2%), 0/48 scheduled (0.0%)
Route 115: 25/53 empirical (47.2%), 28/53 interpolated (52.8%), 0/53 scheduled (0.0%)
Route 119: 31/48 empirical (64.6%), 17/48 interpolated (35.4%), 0/48 scheduled (0.0%)
Route 12: 49/63 empirical (77.8%), 14/63 interpolated (22

In [103]:
i = np.random.choice(len(route_stop_sequences))
print(f"Sample route_id: {sample_route}")
sample_route = list(route_stop_sequences.keys())[i]
route_stop_sequences[sample_route]

Sample route_id: J14


,stop_sequence,stop_id,next_stop_id,t_avg_to_next_min,source
0,0,4470,4450.0,2.500000,empirical
1,1,4450,4451.0,1.030596,interpolated
2,2,4451,15118.0,0.450886,interpolated
3,3,15118,4452.0,0.386473,interpolated
4,4,4452,4453.0,0.354267,interpolated
...,...,...,...,...,...
71,71,5197,5198.0,0.666667,scheduled
72,72,5198,5199.0,0.766667,scheduled
73,73,5199,5200.0,0.766667,scheduled
74,74,5200,5041.0,0.683333,scheduled


### Pickling

In [104]:
# save as pickle
with open("Datasets/route_stop_sequences.pkl", "wb") as f:
    pickle.dump(route_stop_sequences, f)

'''
to load

with open("Datasets/route_stop_sequences.pkl", "rb") as f:
    route_stop_sequences = pickle.load(f)

'''


'\nto load\n\nwith open("Datasets/route_stop_sequences.pkl", "rb") as f:\n    route_stop_sequences = pickle.load(f)\n\n'